In [1]:
import pandas as pd
import numpy as np
import os




In [2]:
TEST_CSV_PATH = "/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv"
SAMPLE_SUB_PATH = (
    "/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

test_df = pd.read_csv(TEST_CSV_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)




In [3]:
test_df["Weeks"] = pd.to_numeric(test_df["Weeks"], errors="coerce")

baseline_df = (
    test_df.sort_values("Weeks")
    .groupby("Patient", as_index=False)
    .first()
    .loc[:, ["Patient", "FVC"]]
)

baseline_fvc = dict(zip(baseline_df["Patient"], baseline_df["FVC"]))




In [4]:
sample_sub["Patient"] = sample_sub["Patient_Week"].str.extract(r"^(.*)_\d+$")[0]

median_fvc = np.median(list(baseline_fvc.values()))
sample_sub["FVC"] = sample_sub["Patient"].map(baseline_fvc).fillna(median_fvc)

# Increase confidence to reduce penalty from Δ/σ; 300 is closer to the optimal σ≈√2·Δ
sample_sub["Confidence"] = 300.0

submission = sample_sub[["Patient_Week", "FVC", "Confidence"]]




In [5]:
OUTPUT_PATH = "/kaggle/working/submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Submission file written to {OUTPUT_PATH} – rows: {submission.shape[0]}")

Submission file written to /kaggle/working/submission.csv – rows: 1908
